# HAR Stage 20A — Full-Screen Homologous Motif Contrast, Conditional-Null Benchmark

**Goal.** Continue the original *perturbation-grounded HAR regulatory benchmarking* study after the Stage 19B source audit. Run **all 152 provisional HAR ortholog pairs × 6 JASPAR2024 TFs**. No manuscript generation, no docking, no Google Drive required.

**Why this step?** The previous Stage 18 six-candidate tests were postselected and produced no q<0.05. This notebook instead tests a **HAR-level six-motif family maximum** for all 152 pairs, then applies BH across all 152 families. The underlying original experimental HAR BED is **still not recovered**; current sequences are from a checksum-locked third-party catalog.

**Method at a glance**

- Globally align human/chimp sequences, apply stored orientation, and score **the same homologous, gap-free windows** on both strands.
- Relative log-odds PWM score computed with pseudocount 0.001 and uniform A/C/G/T background. Exploratory threshold: 0.90.
- Observed test statistic per HAR: largest absolute human–chimp motif score difference among windows scoring ≥0.90 in at least one species, maximized over all six TFs.
- Conditional null: move observed reference-to-query base substitutions to random positions with matching chimp reference bases, holding indel alignment fixed. Run `4999` Monte Carlo permutations per HAR, calculate (+1)-corrected one-sided p for the HAR-level maximum, then BH across 152 HARs.
- This is **not** FIMO-calibrated significance, a dinucleotide-preserving null, independent phylogenetic inference or a non-HAR genomic negative reference. No TF binding, enhancer activity, or causal gene target is measured.

### Instructions

Simply select **Runtime → Run all**. The notebook embeds required FASTA inputs, source audit metadata and Python source. It downloads **only** the pinned JASPAR2024 motif text, and verifies its SHA256. At completion it exports `HAR_Stage20A_FullScreen_Results.zip`. Upload that ZIP to this chat for the next audit.


In [ ]:
# 1 — Extract embedded, checksum-provenanced original Stage 19A ortholog inputs and analysis code
import io, base64, zipfile, os, subprocess, sys, hashlib
from pathlib import Path
root=Path('/content') if Path('/content').exists() else Path.cwd()
PACK="""..."""
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PACK))) as z:
    for name in z.namelist():
        dest=root/name
        dest.write_bytes(z.read(name))
        print('Prepared',name,'bytes',dest.stat().st_size,'SHA256',hashlib.sha256(dest.read_bytes()).hexdigest()[:12])
print('No Google Drive used.')


In [ ]:
# 2 — Install small public dependencies (Colab normally has numpy/requests)
import sys, subprocess
subprocess.run([sys.executable,'-m','pip','install','-q','biopython>=1.84','requests>=2.30','numpy>=1.24'],check=True)
print('Ready for full-screen conditional positional-null test.')


In [ ]:
# 3 — Execute all 152 locus-level conditional nulls (4999 Monte Carlo permutations per locus)
# Change N_PERM only for diagnostic testing. For final publication analyses use the prespecified 4999.
import os,sys,subprocess
os.environ['HAR_N_PERM']='4999'
run=subprocess.run([sys.executable,str(root/'HAR_Stage20A_ConditionalNull_NoDrive.py')],cwd=str(root))
if run.returncode != 0:
    raise RuntimeError(f'Stage 20A did not finish successfully (exit code {run.returncode})')
print('Completed. Download the generated HAR_Stage20A_FullScreen_Results.zip if download did not start automatically.')
